# EPI Investor Demo (`v2.8.4`)

This notebook is designed to show the **real EPI product story** around a high-risk AI-assisted financial decision.

What this demo represents in the real world:
- a **risk or compliance owner** defines the policy rulebook once
- an **AI-assisted financial decision flow** runs and EPI records what actually happened
- EPI's **fault analyzer** checks the run against the rulebook
- a **human reviewer** decides whether the flagged decision should be trusted
- if anyone changes the artifact later, EPI shows it as **tampered**

What the notebook will create:
- `epi_policy.json` -> the company rulebook for this kind of decision
- `investor_demo.epi` -> the original recorded case file
- `analysis.json` inside the artifact -> EPI's explanation of what went wrong
- `review.json` inside the reviewed artifact -> the human review decision
- `investor_demo_tampered.epi` -> a forged copy to prove tamper detection

**Run cells top to bottom.**


In [ ]:
# @title 1. Install EPI Recorder { display-mode: "form" }
INSTALL_SOURCE = "pypi"  # @param ["pypi", "github"]
GITHUB_REF = "main"

import subprocess
import sys

if INSTALL_SOURCE == "pypi":
    install_target = "epi-recorder==2.8.4"
else:
    install_target = f"git+https://github.com/mohdibrahimaiml/epi-recorder.git@{GITHUB_REF}"

cmd = [
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--force-reinstall",
    "--no-cache-dir",
    "cryptography<44",
    "rich<14",
    "pydantic<=2.12.3",
    "jedi>=0.16",
    install_target,
]
result = subprocess.run(cmd, capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Dependency install failed")

import epi_core
print(f"[OK] Installed epi-recorder {epi_core.__version__}")
print("This demo focuses on policy, financial decision review, human review, and evidence trust.")


In [ ]:
# @title 2. Admin Defines the Policy Rulebook { display-mode: "form" }
import json
import shutil
from pathlib import Path

workspace = Path("/content/epi_investor_demo")
workspace.mkdir(parents=True, exist_ok=True)

for stale in [
    workspace / "investor_demo.epi",
    workspace / "investor_demo_reviewed.epi",
    workspace / "investor_demo_tampered.epi",
    workspace / "investor_demo.py",
    workspace / "epi_policy.json",
]:
    if stale.exists():
        if stale.is_dir():
            shutil.rmtree(stale)
        else:
            stale.unlink()

policy = {
    "system_name": "high-risk-funds-approval-agent",
    "system_version": "2.8.4-demo",
    "policy_version": "2026-03-18",
    "profile_id": "finance.loan-underwriting",
    "rules": [
        {
            "id": "R001",
            "name": "Do Not Exceed Balance",
            "severity": "critical",
            "description": "The agent must not approve an amount above the known balance.",
            "type": "constraint_guard",
            "watch_for": ["balance", "available_balance", "limit"],
            "violation_if": "approved_amount > balance"
        },
        {
            "id": "R002",
            "name": "Verify Identity Before Refund",
            "severity": "high",
            "description": "Identity verification must happen before refund.",
            "type": "sequence_guard",
            "required_before": "refund",
            "must_call": "verify_identity"
        },
        {
            "id": "R003",
            "name": "Human Approval Above 10000",
            "severity": "high",
            "description": "Amounts above 10,000 require human approval.",
            "type": "threshold_guard",
            "watch_for": ["amount", "approved_amount", "loan_amount"],
            "threshold_value": 10000,
            "required_action": "human_approval"
        },
        {
            "id": "R004",
            "name": "Never Output Secrets",
            "severity": "critical",
            "description": "The system must not expose secret-like tokens in output.",
            "type": "prohibition_guard",
            "pattern": "sk-[A-Za-z0-9]+"
        }
    ]
}

policy_path = workspace / "epi_policy.json"
policy_path.write_text(json.dumps(policy, indent=2), encoding="utf-8")

print("WHAT IS EPI POLICY IN SIMPLE TERMS?")
print("- It is the company rulebook for this kind of AI decision")
print("- It is usually defined once by a risk, compliance, or product owner")
print("- Normal employees should not be writing JSON by hand every day")
print(f"\nCreated policy file: {policy_path}")
print("\nRules now active for this demo decision:")
for rule in policy["rules"]:
    print(f"- {rule['id']}: {rule['name']} ({rule['severity']})")
print("\nEPI will embed this same rulebook into the artifact as policy.json so reviewers know which rules were active at run time.")


In [ ]:
# @title 3. Run the AI-Assisted Financial Decision and Create the Main .epi Artifact { display-mode: "form" }
agent_code = r'''
from pathlib import Path
from epi_recorder import record

output = Path("/content/epi_investor_demo/investor_demo.epi")

with record(
    str(output),
    workflow_name="Investor Demo High-Risk Funds Approval",
    goal="Audit whether an AI-assisted high-risk financial decision should be trusted",
    notes="Intentional policy and heuristic violations for investor demonstration",
    metadata_tags=["investor-demo", "finance", "policy", "fault-analysis"],
) as epi:
    epi.log_step("tool.response", {
        "tool": "fetch_account_context",
        "account_id": "ACC-4401",
        "customer_id": "CUST-9001",
        "balance": 500.0,
        "currency": "USD",
    })

    epi.log_step("llm.error", {
        "error": "Rate limit exceeded while fetching sanctions data",
        "provider": "demo-llm",
    })

    epi.log_step("tool.call", {
        "tool": "approve_loan_disbursement",
        "amount": 2000.0,
        "account_id": "ACC-4401",
        "reason": "urgent disbursement",
    })

    epi.log_step("tool.call", {
        "tool": "process_refund",
        "amount": 200.0,
        "reference": "REF-2026-001",
    })

    epi.log_step("tool.call", {
        "tool": "approve_wire_transfer",
        "amount": 15000.0,
        "destination": "vendor-settlement",
    })

    epi.log_step("llm.response", {
        "text": "Escalation note: temporary key observed as sk-ABC123SECRET for retry flow.",
        "model": "demo-llm",
    })

    epi.log_step("tool.response", {
        "tool": "create_case",
        "case_reference": "CASE-7702",
        "status": "pending_manual_review",
    })

print(f"Created artifact: {output}")
'''

(workspace / "investor_demo.py").write_text(agent_code, encoding="utf-8")

%cd /content/epi_investor_demo
!python investor_demo.py

epi_file = workspace / "investor_demo.epi"
assert epi_file.exists(), "Expected investor_demo.epi to be created"
print(f"\n[PACKAGE] {epi_file.name} ({epi_file.stat().st_size // 1024} KB)")
print("This artifact now contains the decision trace plus policy-grounded analysis before sealing.")


In [ ]:
# @title 4. EPI Fault Analyzer Checks the Decision Against the Policy { display-mode: "form" }
import json
import sys
import zipfile
from pathlib import Path

epi_file = Path("/content/epi_investor_demo/investor_demo.epi")
assert epi_file.exists(), "Run the previous cell first"

print("=" * 70)
print("VERIFY ARTIFACT")
!{sys.executable} -m epi_cli.main verify {epi_file}

print("\n" + "=" * 70)
print("FAULT ANALYZER OUTPUT")
!{sys.executable} -m epi_cli.main analyze {epi_file}

with zipfile.ZipFile(epi_file, "r") as zf:
    policy_json = json.loads(zf.read("policy.json").decode("utf-8"))
    analysis_json = json.loads(zf.read("analysis.json").decode("utf-8"))

primary_fault = analysis_json.get("primary_fault", {})
print("\nHOW DID EPI USE THE POLICY?")
print(f"- Rules embedded in artifact: {len(policy_json.get('rules', []))}")
print(f"- Primary fault type: {primary_fault.get('fault_type')}")
print(f"- Violated rule: {primary_fault.get('rule_id')}")
print(f"- Exact step: {primary_fault.get('step_index')}")
print(f"- Why it matters: {primary_fault.get('why_it_matters')}")
print("\nThis is where EPI goes beyond recording: it compares the recorded decision against the rulebook and explains exactly where the decision became unsafe or non-compliant.")


In [ ]:
# @title 5. Human Reviewer Confirms the Fault Without Rewriting History { display-mode: "form" }
import shutil
from epi_cli.keys import KeyManager, generate_default_keypair_if_missing
from epi_core.review import ReviewRecord, add_review_to_artifact, make_review_entry

epi_file = Path("/content/epi_investor_demo/investor_demo.epi")
reviewed = Path("/content/epi_investor_demo/investor_demo_reviewed.epi")
if reviewed.exists():
    reviewed.unlink()
shutil.copy2(epi_file, reviewed)

with zipfile.ZipFile(epi_file, "r") as zf:
    analysis_json = json.loads(zf.read("analysis.json").decode("utf-8"))

generate_default_keypair_if_missing(console_output=False)
review = ReviewRecord(
    reviewed_by="investor.demo.reviewer@epilabs.org",
    reviews=[
        make_review_entry(
            fault=analysis_json["primary_fault"],
            outcome="confirmed_fault",
            notes="Confirmed for demo: this violation was intentionally introduced to show EPI review.",
            reviewer="investor.demo.reviewer@epilabs.org",
        )
    ],
)
review.sign(KeyManager().load_private_key("default"))
add_review_to_artifact(reviewed, review)

with zipfile.ZipFile(reviewed, "r") as zf:
    review_json = json.loads(zf.read("review.json").decode("utf-8"))

print("HOW DOES THE HUMAN IN THE LOOP WORK?")
print("- A reviewer, manager, auditor, or compliance operator looks at the flagged case")
print("- They do not rewrite the original run or delete the evidence")
print("- They append their decision as review.json so the case file keeps both machine analysis and human judgment")
print(f"\n[REVIEW] Added review.json to {reviewed.name}")
print(f"- Reviewed by: {review_json.get('reviewed_by')}")
print(f"- Outcome: {review_json.get('reviews', [{}])[0].get('outcome')}")
print(f"- Notes: {review_json.get('reviews', [{}])[0].get('notes')}")


In [ ]:
# @title 6. Tamper Test - Forge the Reviewed Artifact { display-mode: "form" }
tampered = Path("/content/epi_investor_demo/investor_demo_tampered.epi")
source_epi = Path("/content/epi_investor_demo/investor_demo_reviewed.epi")
assert source_epi.exists(), "Run the review cell first"
if tampered.exists():
    tampered.unlink()

with zipfile.ZipFile(source_epi, "r") as src, zipfile.ZipFile(tampered, "w", compression=zipfile.ZIP_DEFLATED) as dst:
    for item in src.infolist():
        data = src.read(item.filename)
        if item.filename == "analysis.json":
            payload = json.loads(data.decode("utf-8"))
            payload["fault_detected"] = False
            payload.setdefault("summary", {})
            payload["summary"]["headline"] = "FORGED: analysis changed after sealing"
            data = json.dumps(payload, indent=2).encode("utf-8")
        dst.writestr(item, data)

print("=" * 60)
print("SIGNED REVIEWED ARTIFACT:")
!{sys.executable} -m epi_cli.main verify {source_epi}
print("\n" + "=" * 60)
print("FORGED ARTIFACT:")
!{sys.executable} -m epi_cli.main verify {tampered}
print("=" * 60)
print("\nImportant: a forged file can still contain a signature string, but failed integrity means the artifact is tampered and should not be trusted.")


In [ ]:
# @title 7. Open One Artifact in a Colab-Safe Verified View { display-mode: "form" }
VIEW_TARGET = "reviewed"  # @param ["reviewed", "tampered"]

import html
from IPython.display import HTML, display
from epi_cli.view import _build_viewer_context

generated_epi = Path("/content/epi_investor_demo/investor_demo.epi")
reviewed_epi = Path("/content/epi_investor_demo/investor_demo_reviewed.epi")
tampered_epi = Path("/content/epi_investor_demo/investor_demo_tampered.epi")

def load_artifact(epi_file):
    with zipfile.ZipFile(epi_file) as z:
        names = set(z.namelist())
        manifest = json.loads(z.read("manifest.json"))
        steps = [json.loads(line) for line in z.read("steps.jsonl").decode("utf-8").splitlines() if line.strip()]
        analysis = json.loads(z.read("analysis.json").decode("utf-8")) if "analysis.json" in names else None
        policy = json.loads(z.read("policy.json").decode("utf-8")) if "policy.json" in names else None
        review = json.loads(z.read("review.json").decode("utf-8")) if "review.json" in names else None
        members = sorted(names)
    return manifest, steps, analysis, policy, review, members

def trust_meta(epi_file):
    trust = _build_viewer_context(epi_file)
    label = "Tampered" if not trust.get("integrity_ok") else ("Signed" if trust.get("signature_valid") else "Unsigned")
    color = "#dc2626" if label == "Tampered" else ("#10b981" if label == "Signed" else "#d97706")
    return trust, label, color

def step_summary(step):
    content = step.get("content", {})
    if isinstance(content, str):
        return content
    try:
        return json.dumps(content, ensure_ascii=False)
    except Exception:
        return str(content)

def render_case_view(epi_file, label):
    manifest, steps, analysis, policy, review, members = load_artifact(epi_file)
    trust, trust_label, color = trust_meta(epi_file)
    primary_fault = (analysis or {}).get("primary_fault") or {}
    review_entry = ((review or {}).get("reviews") or [{}])[0]
    review_outcome = review_entry.get("outcome", "Not reviewed")
    review_notes = review_entry.get("notes", "No review notes")
    rule_items = "".join(
        f'<li style="margin:6px 0;"><strong>{html.escape(rule.get("id", "?"))}</strong> - {html.escape(rule.get("name", "Unnamed rule"))}</li>'
        for rule in (policy or {}).get("rules", [])
    ) or '<li>No policy embedded</li>'
    step_items = "".join(
        f'''<div style="padding:10px 0; border-top:1px solid #e5e7eb;">
            <div style="font-weight:700; color:#111827;">#{idx} {html.escape(step.get("kind", "unknown"))}</div>
            <div style="font-size:13px; color:#6b7280; margin:4px 0;">{html.escape(str(step.get("ts", "")))}</div>
            <div style="font-family:ui-monospace, SFMono-Regular, monospace; font-size:12px; color:#1f2937; white-space:pre-wrap;">{html.escape(step_summary(step))}</div>
        </div>'''
        for idx, step in enumerate(steps[:10])
    )
    member_items = "".join(
        f'<li style="margin:6px 0; font-family:ui-monospace, SFMono-Regular, monospace; font-size:12px; color:#4b5563;">{html.escape(name)}</li>'
        for name in members
    )
    return f'''
    <div style="font-family:system-ui, -apple-system, sans-serif; color:#111827; background:#f8fafc; border:1px solid #e5e7eb; border-radius:18px; overflow:hidden; margin:16px 0;">
      <div style="background:{color}; color:white; padding:16px 20px;">
        <div style="font-size:12px; font-weight:700; letter-spacing:0.05em; text-transform:uppercase; opacity:0.9;">{html.escape(label)}</div>
        <div style="font-size:28px; font-weight:800; margin-top:6px;">{html.escape(trust_label)}</div>
        <div style="margin-top:6px; font-size:14px; opacity:0.95;">{html.escape(trust.get("trust_message", "Verification context unavailable"))}</div>
      </div>
      <div style="padding:20px; display:grid; grid-template-columns:1.1fr 0.9fr; gap:18px;">
        <div>
          <div style="background:white; border:1px solid #e5e7eb; border-radius:14px; padding:16px; margin-bottom:16px;">
            <div style="font-size:12px; font-weight:700; letter-spacing:0.04em; text-transform:uppercase; color:#6b7280;">Selected .epi file</div>
            <div style="font-size:18px; font-weight:800; margin-top:8px;">{html.escape(epi_file.name)}</div>
            <div style="margin-top:8px; color:#4b5563; font-family:ui-monospace, SFMono-Regular, monospace; font-size:12px;">{html.escape(str(epi_file))}</div>
            <div style="margin-top:8px; color:#4b5563;">Size: <strong>{html.escape(str(epi_file.stat().st_size // 1024))} KB</strong></div>
          </div>
          <div style="background:white; border:1px solid #e5e7eb; border-radius:14px; padding:16px; margin-bottom:16px;">
            <div style="font-size:12px; font-weight:700; letter-spacing:0.04em; text-transform:uppercase; color:#6b7280;">Recording Goal</div>
            <div style="font-size:20px; font-weight:800; margin-top:8px;">{html.escape(manifest.get("goal", "Execution evidence"))}</div>
            <div style="margin-top:10px; color:#4b5563;">Workflow ID: {html.escape(manifest.get("workflow_id", "unknown"))}</div>
          </div>
          <div style="background:white; border:1px solid #e5e7eb; border-radius:14px; padding:16px; margin-bottom:16px;">
            <div style="font-size:12px; font-weight:700; letter-spacing:0.04em; text-transform:uppercase; color:#6b7280;">Fault Analysis</div>
            <div style="font-size:18px; font-weight:800; margin-top:8px;">{html.escape((analysis or {}).get("summary", {}).get("headline", primary_fault.get("plain_english", "No primary fault embedded")))}</div>
            <div style="margin-top:10px; color:#4b5563;">Why it matters: {html.escape(primary_fault.get("why_it_matters", "Unavailable"))}</div>
            <div style="margin-top:10px; color:#4b5563;">Rule: {html.escape(primary_fault.get("rule_id", "Unavailable"))} | Step: {html.escape(str(primary_fault.get("step_number", primary_fault.get("step_index", "?"))))}</div>
          </div>
          <div style="background:white; border:1px solid #e5e7eb; border-radius:14px; padding:16px;">
            <div style="font-size:12px; font-weight:700; letter-spacing:0.04em; text-transform:uppercase; color:#6b7280;">Execution Timeline</div>
            <div style="margin-top:8px; color:#4b5563;">Showing first {min(len(steps), 10)} of {len(steps)} recorded steps.</div>
            <div style="margin-top:8px;">{step_items}</div>
          </div>
        </div>
        <div>
          <div style="background:white; border:1px solid #e5e7eb; border-radius:14px; padding:16px; margin-bottom:16px;">
            <div style="font-size:12px; font-weight:700; letter-spacing:0.04em; text-transform:uppercase; color:#6b7280;">Trust Facts</div>
            <div style="margin-top:8px; color:#4b5563;">Integrity: <strong>{html.escape('Intact' if trust.get('integrity_ok') else 'Compromised')}</strong></div>
            <div style="margin-top:6px; color:#4b5563;">Signature: <strong>{html.escape('Valid' if trust.get('signature_valid') else ('Present' if trust.get('has_signature') else 'Absent'))}</strong></div>
            <div style="margin-top:6px; color:#4b5563;">Files in manifest: <strong>{html.escape(str(trust.get('files_checked', 'Unknown')))}</strong></div>
            <div style="margin-top:6px; color:#4b5563;">Mismatches: <strong>{html.escape(str(trust.get('mismatches_count', 0)))}</strong></div>
          </div>
          <div style="background:white; border:1px solid #e5e7eb; border-radius:14px; padding:16px; margin-bottom:16px;">
            <div style="font-size:12px; font-weight:700; letter-spacing:0.04em; text-transform:uppercase; color:#6b7280;">Contents of this .epi file</div>
            <ul style="margin:10px 0 0 18px; padding:0;">{member_items}</ul>
          </div>
          <div style="background:white; border:1px solid #e5e7eb; border-radius:14px; padding:16px; margin-bottom:16px;">
            <div style="font-size:12px; font-weight:700; letter-spacing:0.04em; text-transform:uppercase; color:#6b7280;">Rules In Force</div>
            <div style="margin-top:8px; font-weight:700;">{html.escape((policy or {}).get('system_name', 'No embedded policy'))}</div>
            <ul style="margin:10px 0 0 18px; color:#4b5563;">{rule_items}</ul>
          </div>
          <div style="background:white; border:1px solid #e5e7eb; border-radius:14px; padding:16px;">
            <div style="font-size:12px; font-weight:700; letter-spacing:0.04em; text-transform:uppercase; color:#6b7280;">Human Review</div>
            <div style="margin-top:8px; color:#4b5563;">Outcome: <strong>{html.escape(review_outcome)}</strong></div>
            <div style="margin-top:6px; color:#4b5563;">Reviewed by: <strong>{html.escape((review or {}).get('reviewed_by', 'Unavailable'))}</strong></div>
            <div style="margin-top:6px; color:#4b5563;">Notes: {html.escape(review_notes)}</div>
          </div>
        </div>
      </div>
    </div>
    '''

target_map = {
    "reviewed": (reviewed_epi, "Happy path: reviewed artifact"),
    "tampered": (tampered_epi, "Tamper test: forged artifact"),
}
selected_epi, selected_label = target_map[VIEW_TARGET]
trust, trust_label, color = trust_meta(selected_epi)
print(f"Selected artifact: {selected_epi.name}")
print(f"Computed trust from verification context: {trust_label}")
print(f"- integrity_ok: {trust.get('integrity_ok')}")
print(f"- signature_valid: {trust.get('signature_valid')}")
print(f"- mismatches_count: {trust.get('mismatches_count')}")
display(HTML(render_case_view(selected_epi, selected_label)))

print("What the investor should notice:")
if VIEW_TARGET == "reviewed":
    print("- This is the full case file: recorded decision + policy + fault analysis + human review")
    print("- It shows what happened, what rule was broken, and how a human reviewer responded")
else:
    print("- This is the forged copy after someone changed analysis.json")
    print("- It shows why signed evidence still needs integrity checks")
print("- The original generated file is the sealed output EPI produced at run time")
print("- The panel above is rendered directly from the selected .epi file and its verified trust context")

try:
    from google.colab import files
    files.download(str(selected_epi))
except Exception:
    pass
